# EDA - Store Sales Time Series Forecasting

Cilj: razumjeti podatke prije feature engineeringa i modeliranja.

Fajlovi:
- `train.csv` - dnevna prodaja po prodavnici i kategoriji (2013-2017)
- `stores.csv` - metadata prodavnica
- `transactions.csv` - broj transakcija po danu
- `oil.csv` - dnevna cijena nafte
- `holidays_events.csv` - praznici i eventi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)

DATA = '../data/raw/'
EARTHQUAKE_DATE = pd.Timestamp('2016-04-16')

## 1. Ucitavanje podataka

In [ ]:
train        = pd.read_csv(DATA + 'train.csv', parse_dates=['date'])
stores       = pd.read_csv(DATA + 'stores.csv')
transactions = pd.read_csv(DATA + 'transactions.csv', parse_dates=['date'])
oil          = pd.read_csv(DATA + 'oil.csv', parse_dates=['date'])
holidays     = pd.read_csv(DATA + 'holidays_events.csv', parse_dates=['date'])

print(f'train:        {train.shape}')
print(f'stores:       {stores.shape}')
print(f'transactions: {transactions.shape}')
print(f'oil:          {oil.shape}')
print(f'holidays:     {holidays.shape}')

## 2. train.csv

In [ ]:
train.head(10)

In [ ]:
train.info()

In [ ]:
print('Missing values:')
print(train.isnull().sum())
print(f'\nPeriod: {train.date.min().date()} do {train.date.max().date()}')
print(f'Prodavnice: {train.store_nbr.nunique()}')
print(f'Kategorije: {train.family.nunique()}')
print(f'Redova sa sales=0: {(train.sales == 0).sum():,} ({(train.sales == 0).mean():.1%})')

In [ ]:
train.sales.describe()

In [ ]:
# Ukupna dnevna prodaja kroz cijeli period
daily = train.groupby('date')['sales'].sum()

fig, ax = plt.subplots()
ax.plot(daily.index, daily.values, linewidth=0.8, color='steelblue')
ax.axvline(EARTHQUAKE_DATE, color='red', linestyle='--', linewidth=1.2, label='Zemljotres (16.04.2016)')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_title('Ukupna dnevna prodaja (sve prodavnice, sve kategorije)')
ax.set_ylabel('Sales')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Prodaja po kategoriji
by_family = train.groupby('family')['sales'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
by_family.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Ukupna prodaja po kategoriji proizvoda')
ax.set_xlabel('')
ax.set_ylabel('Sales')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Sezonalnost: prodaja po danu u sedmici
train['dayofweek'] = train.date.dt.day_name()
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
by_dow = train.groupby('dayofweek')['sales'].mean().reindex(dow_order)

fig, ax = plt.subplots(figsize=(9, 4))
by_dow.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Prosjecna prodaja po danu u sedmici')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Sezonalnost: prodaja po mjesecu
train['month'] = train.date.dt.month
by_month = train.groupby('month')['sales'].mean()

fig, ax = plt.subplots(figsize=(9, 4))
by_month.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Prosjecna prodaja po mjesecu')
ax.set_xlabel('Mjesec')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Efekat promocija
print('Prosjecna prodaja - nula stavki na promociji:', round(train[train.onpromotion == 0]['sales'].mean(), 2))
print('Prosjecna prodaja - barem jedna stavka na promociji:', round(train[train.onpromotion > 0]['sales'].mean(), 2))

## 3. stores.csv

In [ ]:
stores.head()

In [ ]:
print('Tipovi prodavnica:', stores['type'].value_counts().to_dict())
print('Gradovi:', stores['city'].nunique())
print('Clusteri:', stores['cluster'].nunique())
stores.describe()

In [ ]:
# Prodaja po tipu prodavnice
train_stores = train.merge(stores, on='store_nbr')
by_type = train_stores.groupby('type')['sales'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
by_type.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Prosjecna prodaja po tipu prodavnice')
ax.set_xlabel('Tip')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. transactions.csv

In [ ]:
transactions.head()

In [ ]:
# Korelacija transakcija i prodaje
daily_sales = train.groupby('date')['sales'].sum().reset_index()
daily_trans = transactions.groupby('date')['transactions'].sum().reset_index()
merged = daily_sales.merge(daily_trans, on='date')

corr = merged['sales'].corr(merged['transactions'])
print(f'Korelacija (sales vs transactions): {corr:.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(merged['transactions'], merged['sales'], alpha=0.3, s=8, color='steelblue')
ax.set_title(f'Transakcije vs Prodaja (r={corr:.2f})')
ax.set_xlabel('Transakcije')
ax.set_ylabel('Sales')
plt.tight_layout()
plt.show()

## 5. oil.csv

In [ ]:
oil.head()

In [ ]:
print(f'Missing values: {oil.dcoilwtico.isnull().sum()} ({oil.dcoilwtico.isnull().mean():.1%})')
print(f'Period: {oil.date.min().date()} do {oil.date.max().date()}')

# Vikendi i praznici nemaju cijenu - interpoliramo linearno
oil_full = oil.set_index('date').reindex(
    pd.date_range(oil.date.min(), oil.date.max())
).interpolate().reset_index()
oil_full.columns = ['date', 'dcoilwtico']

fig, ax = plt.subplots()
ax.plot(oil_full.date, oil_full.dcoilwtico, linewidth=0.9, color='darkorange')
ax.set_title('Cijena nafte (WTI) - interpolovano za vikende i praznike')
ax.set_ylabel('USD/bbl')
plt.tight_layout()
plt.show()

In [ ]:
# Korelacija cijene nafte i ukupne prodaje
merged_oil = merged.merge(oil_full, on='date', how='left')
corr_oil = merged_oil['sales'].corr(merged_oil['dcoilwtico'])
print(f'Korelacija (sales vs cijena nafte): {corr_oil:.3f}')

## 6. holidays_events.csv

Vazno: `transferred=True` znaci da je taj dan premjesten i u stvari je normalan radni dan.
Stvarni praznik se nalazi u redu gdje je `type='Transfer'`.

In [ ]:
holidays.head(10)

In [ ]:
print('Tipovi:', holidays['type'].value_counts().to_dict())
print('Locale:', holidays['locale'].value_counts().to_dict())
print('Transferred:', holidays['transferred'].value_counts().to_dict())

# Primjer transferovanog praznika
print('\nPrimjer - transferovani praznici:')
print(holidays[holidays['transferred'] == True].head(3)[['date','description','type','transferred']])
print('\nNjihovi stvarni datumi (type=Transfer):')
print(holidays[holidays['type'] == 'Transfer'].head(3)[['date','description','type']])

In [ ]:
# Ispravno: stvarni praznici su oni koji NISU transferred + oni gdje je type='Transfer'
real_holidays = holidays[
    (holidays['transferred'] == False) & (holidays['type'] != 'Bridge')
]

national_holidays = real_holidays[real_holidays['locale'] == 'National']['date'].unique()

daily2 = train.groupby('date')['sales'].sum().reset_index()
daily2['is_national_holiday'] = daily2['date'].isin(national_holidays)

effect = daily2.groupby('is_national_holiday')['sales'].mean()
print('Prosjecna dnevna prodaja (nacionalni praznici):')
print(f'  Obican dan:        {effect[False]:,.0f}')
print(f'  Nacionalni praznik:{effect[True]:,.0f}')
print(f'  Razlika:           {(effect[True]/effect[False]-1):+.1%}')

## 7. Posebni eventi

### 7a. Efekat isplate plata (payday)

Plate u javnom sektoru isplacuju se 15. i zadnjeg dana u mjesecu.

In [ ]:
daily3 = train.groupby('date')['sales'].sum().reset_index()
daily3['day'] = daily3['date'].dt.day
daily3['days_in_month'] = daily3['date'].dt.days_in_month
daily3['is_payday'] = (daily3['day'] == 15) | (daily3['day'] == daily3['days_in_month'])

payday_effect = daily3.groupby('is_payday')['sales'].mean()
print('Prosjecna dnevna prodaja:')
print(f'  Obican dan: {payday_effect[False]:,.0f}')
print(f'  Payday:     {payday_effect[True]:,.0f}')
print(f'  Razlika:    {(payday_effect[True]/payday_effect[False]-1):+.1%}')

### 7b. Efekat zemljotresa (16. april 2016.)

In [ ]:
# Prodaja 4 sedmice prije i 4 sedmice poslije zemljotresa
window_start = EARTHQUAKE_DATE - pd.Timedelta(days=28)
window_end   = EARTHQUAKE_DATE + pd.Timedelta(days=28)

eq_window = daily3[(daily3['date'] >= window_start) & (daily3['date'] <= window_end)]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(eq_window['date'], eq_window['sales'], color='steelblue', linewidth=1)
ax.axvline(EARTHQUAKE_DATE, color='red', linestyle='--', linewidth=1.5, label='Zemljotres (16.04.2016)')
ax.set_title('Prodaja 4 sedmice prije i poslije zemljotresa')
ax.set_ylabel('Sales')
ax.legend()
plt.tight_layout()
plt.show()

before = daily3[(daily3['date'] >= window_start) & (daily3['date'] < EARTHQUAKE_DATE)]['sales'].mean()
after  = daily3[(daily3['date'] > EARTHQUAKE_DATE) & (daily3['date'] <= window_end)]['sales'].mean()
print(f'Prosjecna prodaja prije: {before:,.0f}')
print(f'Prosjecna prodaja poslije: {after:,.0f}')
print(f'Razlika: {(after/before-1):+.1%}')

## 8. Zakljucak i plan za feature engineering

**Sta smo naucili:**
- train ima znacajan udio nula u sales - log1p transformacija za modeliranje
- cijena nafte ima missing values za vikende i praznike - linearna interpolacija
- jasna sedmicna sezonalnost (subota/nedjelja visi) i godisnja sezonalnost
- promocije povecavaju prodaju
- transferred praznici su normalni dani - koristiti samo type='Transfer' i transferred=False redove
- payday efekat (15. i zadnji u mjesecu) vidljiv u podacima
- zemljotres 16.04.2016. napravio anomaliju - treba ga oznaciti kao feature

**Merge plan za master dataset:**
- train + stores (left join na `store_nbr`)
- + transactions (left join na `date` + `store_nbr`)
- + oil interpolovano (left join na `date`)
- + holidays agregirano po datumu: national flag, local flag (po gradu)

**Features za modeliranje:**
- lag features: sales t-7, t-14, t-28 (t-1 nije dostupan za test)
- rolling mean: 7d, 14d, 28d
- kalendarske: dayofweek, month, weekofyear, is_weekend
- is_national_holiday, is_local_holiday (po gradu prodavnice)
- is_payday (15. i zadnji u mjesecu)
- days_after_earthquake (za period nakon 16.04.2016.)
- cijena nafte (interpolovana)
- store tip, cluster
- onpromotion